# 🚀 HUẤN LUYỆN MODEL PP-OCRv4 CHUYÊN BIỆT CHO BẢN VẼ KỸ THUẬT CAD
Notebook này được thiết kế tự động để fine-tune mô hình nhận diện chữ **PP-OCRv4 Recognition** với tập dữ liệu **3,500 ảnh CAD**: dung sai 2 tầng thông thường (`++`, `--`, `0/-`, `+/0`, `+/-`), vạch gạch ngang phân số, và các lỗi gõ thừa dấu cách.

- **Môi trường**: Google Colab (GPU T4 miễn phí)
- **Thời gian huấn luyện**: ~10 - 15 phút (35 epochs)
- **Đầu ra**: Mô hình Inference (`.pdmodel` và `.onnx`) tương thích 100% với **RapidOCR** để thay thế model gốc.

In [ ]:
# BƯỚC 1: KIỂM TRA GPU
# Đảm bảo bạn đã chọn Runtime > Change runtime type > T4 GPU
!nvidia-smi

In [ ]:
# BƯỚC 2: CÀI ĐẶT PADDLEPADDLE GPU VÀ MÔI TRƯỜNG (CHO COLAB CUDA 12)
%cd /content

# 1. Cài đặt PaddlePaddle GPU 3.2.0 từ kho cu126 (tương thích 100% Colab T4 GPU)
!pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# 2. Cài đặt onnx và paddle2onnx từ kho PyPI chính thức
!pip install -i https://pypi.org/simple onnx paddle2onnx

# 3. Clone PaddleOCR nếu chưa có
import os
if not os.path.exists('/content/PaddleOCR'):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR

%cd /content/PaddleOCR
!pip install -r requirements.txt

# 4. Kiểm tra GPU
import paddle
print('🎉 CÀI ĐẶT THÀNH CÔNG! Paddle version:', paddle.__version__)
print('🔥 GPU CUDA sẵn sàng:', paddle.device.is_compiled_with_cuda())


In [ ]:
# BƯỚC 3: TẢI LÊN VÀ GIẢI NÉN BỘ DỮ LIỆU TỪ MÁY TÍNH
import os, zipfile
from google.colab import files

# 1. Xóa file rác/file lỗi cũ nếu có
!rm -f /content/synthetic_cad_dataset.zip
!rm -rf /content/PaddleOCR/synthetic_cad_dataset

# 2. Hiện nút tải file từ máy tính
print('👉 Bấm nút [Choose Files] bên dưới để chọn file synthetic_cad_dataset.zip từ máy:')
uploaded = files.upload()

# 3. Xác định tên file tải lên và giải nén
zip_path = None
for fn in uploaded.keys():
    if fn.endswith('.zip'):
        zip_path = fn
        break

if zip_path:
    print(f'Đang giải nén {zip_path} ({os.path.getsize(zip_path)} bytes) vào PaddleOCR...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/PaddleOCR/')
    print('✅ ĐÃ GIẢI NÉN THÀNH CÔNG! Danh sách các file:')
    !ls -la /content/PaddleOCR/synthetic_cad_dataset/
else:
    print('❌ Chưa chọn được file zip. Vui lòng bấm chạy lại ô này và chọn file .zip!')


In [ ]:
# BƯỚC 4: TẢI MÔ HÌNH PRETRAINED PP-OCRv4 (EN_PP-OCRV4_REC)
%cd /content/PaddleOCR
!mkdir -p pretrain_models
!wget -nc https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_train.tar -O pretrain_models/en_PP-OCRv4_rec_train.tar
!tar -xf pretrain_models/en_PP-OCRv4_rec_train.tar -C pretrain_models/
!ls -la pretrain_models/en_PP-OCRv4_rec_train/

In [ ]:
# BƯỚC 5: TẠO FILE CONFIG REC_CAD_V4.YML ĐƯỢC CẤU HÌNH TỐI ƯU
config_content = """
Global:
  use_gpu: true
  epoch_num: 35
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/rec_cad_v4/
  save_epoch_step: 5
  eval_batch_step: [0, 50]
  cal_metric_during_train: true
  pretrained_model: ./pretrain_models/en_PP-OCRv4_rec_train/best_accuracy
  checkpoints:
  save_inference_dir: ./inference/rec_cad_v4
  use_visualdl: false
  infer_img: ./synthetic_cad_dataset/images/
  character_dict_path: ./synthetic_cad_dataset/cad_dict.txt
  max_text_length: 35
  infer_mode: false
  use_space_char: true
  distributed: false
  save_res_path: ./output/rec/predicts_cad.txt

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0005
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 3.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: PPLCNetV3
    scale: 0.95
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 120
            depth: 2
            hidden_dims: 120
            kernel_size: [1, 3]
            use_guide: True
          Head:
            fc_decay: 0.00001
      - NRTRHead:
          nrtr_dim: 384
          max_text_length: 35

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - NRTRLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc
  ignore_space: false

Train:
  dataset:
    name: SimpleDataSet
    data_dir: ./synthetic_cad_dataset/
    label_file_list:
      - ./synthetic_cad_dataset/rec_gt_train.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecAug:
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_gtc
            - length
            - valid_ratio
  loader:
    shuffle: true
    batch_size_per_card: 32
    drop_last: true
    num_workers: 2

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./synthetic_cad_dataset/
    label_file_list:
      - ./synthetic_cad_dataset/rec_gt_val.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_gtc
            - length
            - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 32
    num_workers: 2
"""
with open('./synthetic_cad_dataset/rec_cad_v4.yml', 'w') as f:
    f.write(config_content.strip())
print('✔ Đã cấu hình xong synthetic_cad_dataset/rec_cad_v4.yml')


In [ ]:
# BƯỚC 6: SỬA LỖI TƯƠNG THÍCH NUMPY 2 VÀ BẮT ĐẦU HUẤN LUYỆN
%cd /content/PaddleOCR

# 1. Khôi phục tools/train.py về nguyên bản
!git checkout tools/train.py

# 2. Chèn đoạn vá lỗi vào sau các dòng __future__
patch_code = """
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {'int': [np.int8, np.int16, np.int32, np.int64], 'uint': [np.uint8, np.uint16, np.uint32, np.uint64], 'float': [np.float16, np.float32, np.float64], 'complex': [np.complex64, np.complex128], 'others': [bool, object, bytes, str, np.void]}
if not hasattr(np, 'bool'): np.bool = bool
if not hasattr(np, 'int'): np.int = int
if not hasattr(np, 'float'): np.float = float
"""
with open('tools/train.py', 'r', encoding='utf-8') as f:
    lines = f.readlines()

new_lines = []
inserted = False
for line in lines:
    new_lines.append(line)
    if not inserted and 'from __future__' in line:
        pass
    elif not inserted and line.strip() and not line.startswith('#'):
        new_lines.append(patch_code + '\n')
        inserted = True

with open('tools/train.py', 'w', encoding='utf-8') as f:
    f.writelines(new_lines)

print('✅ Đã định vị chuẩn xác đoạn vá NumPy 2! Bắt đầu huấn luyện...')

# 3. Tiến hành huấn luyện (Train)
!python tools/train.py -c synthetic_cad_dataset/rec_cad_v4.yml


In [ ]:
# BƯỚC 7: ĐÁNH GIÁ ĐỘ CHÍNH XÁC TRÊN TẬP VALIDATION
!python tools/eval.py -c synthetic_cad_dataset/rec_cad_v4.yml -o Global.checkpoints=./output/rec_cad_v4/best_accuracy

In [ ]:
# BƯỚC 8: XUẤT MÔ HÌNH VÀ CHUYỂN ĐỔI SANG ONNX CHO RAPIDOCR / PYTHON LOCAL
%cd /content/PaddleOCR

# 1. Xuất mô hình Paddle Inference chuẩn
!python tools/export_model.py -c synthetic_cad_dataset/rec_cad_v4.yml -o Global.pretrained_model=./output/rec_cad_v4/best_accuracy Global.save_inference_dir=./inference/rec_cad_v4

# 2. Đảm bảo công cụ paddle2onnx sẵn sàng trên Colab Python 3.13
!pip install onnx
!which paddle2onnx || (pip download --no-deps paddle2onnx==2.1.0 && unzip -o paddle2onnx-*.whl -d /tmp/p2o && cp $(find /tmp/p2o -name paddle2onnx -type f) /usr/local/bin/ && chmod +x /usr/local/bin/paddle2onnx)

# 3. Chuyển đổi trực tiếp sang ONNX model (dùng ngay trong RapidOCR)
!paddle2onnx --model_dir ./inference/rec_cad_v4 --model_filename inference.pdmodel --params_filename inference.pdiparams --save_file ./inference/rec_cad_v4/rec_cad_v4.onnx --opset_version 14 --enable_onnx_checker True

# 4. Sao chép từ điển chuẩn
!cp synthetic_cad_dataset/cad_dict.txt ./inference/rec_cad_v4/

# 5. Đóng gói file zip và tự động tải về máy tính
!zip -r /content/paddleocr_cad_finetuned.zip ./inference/rec_cad_v4
print('✔ Đang tải file mô hình paddleocr_cad_finetuned.zip về máy...')
from google.colab import files
files.download('/content/paddleocr_cad_finetuned.zip')
